## Phase 1 : Document Loader

In [9]:
from langchain_community.document_loaders import PyPDFLoader

In [10]:
loader = PyPDFLoader("data/doc1.pdf")
documents = loader.load()

In [11]:
print("Pages : ", len(documents))

Pages :  2


In [13]:
for doc in documents[:2]:
    print(doc.page_content[:500])

General
 
Disease–Medicine
 
Dataset
 
1.
 
Common
 
Cold
 
○
 
Symptoms:
 
Runny
 
nose,
 
sneezing,
 
sore
 
throat,
 
mild
 
fever
 
○
 
Common
 
medicines:
 
Paracetamol,
 
saline
 
nasal
 
spray
 
○
 
Note:
 
Usually
 
resolves
 
with
 
rest
 
and
 
fluids.
 
2.
 
Fever
 
○
 
Symptoms:
 
Increased
 
body
 
temperature,
 
chills,
 
weakness
 
○
 
Common
 
medicine:
 
Paracetamol
 
○
 
Note:
 
Dosage
 
depends
 
on
 
age,
 
weight,
 
and
 
medical
 
condition.
 
3.
 
Headache
 
○
 
Symptoms:

11.
 
Migraine
 
○
 
Symptoms:
 
Severe
 
headache,
 
nausea,
 
sensitivity
 
to
 
light/sound
 
○
 
Common
 
medicines:
 
Paracetamol,
 
Ibuprofen
 
○
 
Note:
 
Recurrent
 
migraines
 
may
 
require
 
prescription
 
treatment.
 
12.
 
Motion
 
Sickness
 
○
 
Symptoms:
 
Nausea,
 
dizziness,
 
vomiting
 
during
 
travel
 
○
 
Common
 
medicine:
 
Dimenhydrinate
 
○
 
Note:
 
Some
 
medicines
 
can
 
cause
 
drowsiness.
 
13.
 
Athlete's
 
Foot
 
○
 
Symptoms:
 
Itching,
 
scaling,
 
cracking
 


In [14]:
print(doc.metadata)

{'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'doc1', 'source': 'data/doc1.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}


## Phase 2 : Chunk the document

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

chunks = splitter.split_documents(documents)

print("Original Pages : ", len(documents))
print("Chunks : ", len(chunks))

Original Pages :  2
Chunks :  19


## Phase 3 : Embeddings

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

In [25]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\gagan\Desktop\GenAI\Module 9\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4980.84it/s]


In [26]:
vector = embeddings.embed_query("Which medicine i can take in fever?")
print(vector, len(vector))

[0.01616363413631916, 0.0027985882479697466, -0.011673886328935623, -0.029906949028372765, 0.03569720312952995, -0.03714648634195328, 0.04305082559585571, 0.09554493427276611, 0.03625592216849327, -0.00538985338062048, -0.049680277705192566, -0.03094675950706005, 0.05436200276017189, 0.11893686652183533, -0.007779688108712435, -0.03511351719498634, 0.0032659515272825956, 0.0072827814146876335, 0.02664714679121971, -0.0294363833963871, -0.09812721610069275, 0.013448478654026985, -0.03480483219027519, 0.05694987624883652, -0.007630559150129557, 0.11318245530128479, 0.002675455529242754, 0.053466856479644775, -0.021002154797315598, -0.017951086163520813, 0.014957078732550144, 0.010279431939125061, 0.011645309627056122, -0.01820291019976139, -0.016145307570695877, 0.021217970177531242, -0.09141681343317032, -0.038283929228782654, -0.022303340956568718, 0.06857942789793015, 0.0040484825149178505, 0.0566677562892437, 0.05560543015599251, -0.05770059674978256, 0.06286560744047165, -0.00855183

## Phase 4 : Chroma Database (storing vector)

In [29]:
from langchain_chroma import Chroma

In [32]:
vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)